<a href="https://colab.research.google.com/github/diaoumardia2001-beep/DI-Bootcamp-May/blob/main/Exercises_XP_VDB_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Vector Databases and RAG
Use this guided notebook and fill each TODO before running cells.

## What you'll learn
- Vector search strategies (KNN, ANN) and evaluation.
- Vector database utility (similarity search, RAG).
- Differences between vector DBs, libraries, and plugins.
- Best practices for vector store usage and performance.
- How LMs use context; embedding generation and storage.
- Querying vector stores and applying LMs for QA with retrieved context.

## What you'll build
A functional RAG pipeline with FAISS and ChromaDB, plus QA over retrieved context using a Hugging Face model.

## 0. Setup
Run the install cell once. If your platform needs system deps (e.g., libomp for FAISS), follow instructions in comments.

In [ ]:
%pip uninstall -y pydantic-core pydantic
%pip install -U "pydantic<2"
%pip install -U "faiss-cpu>=1.8.0" "chromadb==0.3.21"
%pip install -U "numpy<2" sentence-transformers transformers

In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from IPython.display import display
os.makedirs('cache', exist_ok=True)


## 🌟 Exercise 1 · Data loading and preparation

In [ ]:
import pandas as pd

data_path = 'labelled_newscatcher_dataset.csv'
pdf = pd.read_csv(data_path, sep=';')

# 1. Vérification et création de la colonne d'identifiant unique 'id'
if 'id' not in pdf.columns:
    pdf['id'] = range(len(pdf))

display(pdf.head())

# 2. Création d'un sous-ensemble maniable (les 1000 premières lignes)
pdf_subset = pdf.head(1000) # ou pdf.iloc[:1000]

# Affichage des premières lignes pour vérifier la présence de 'id' et 'title'
pdf_subset[['id', 'title']].head()

## 🌟 Exercise 2 · Vectorization with Sentence Transformers

In [ ]:
# S'assure que la classe InputExample est importée (généralement depuis sentence_transformers)
from sentence_transformers.readers import InputExample

# Génération des exemples d'entraînement à l'aide d'une compréhension de liste
faiss_train_examples = [
    example_create_fn(row['id'], row['title'])
    for _, row in pdf_subset.iterrows()
]

# Affichage des deux premiers exemples pour vérification
faiss_train_examples[:2]

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
titles_list = pdf_subset['title'].tolist()
faiss_title_embedding = model.encode(titles_list, convert_to_numpy=True, show_progress_bar=True)
len(faiss_title_embedding), len(faiss_title_embedding[0])

## 🌟 Exercise 3 · FAISS indexing and search

In [ ]:
pdf_to_index = pdf_subset
id_index = pdf_to_index['id'].to_numpy().astype(np.int64)
content_encoded_normalized = faiss_title_embedding.astype('float32')
faiss.normalize_L2(content_encoded_normalized)
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(content_encoded_normalized.shape[1]))
index_content.add_with_ids(content_encoded_normalized, id_index)
index_content.ntotal


In [ ]:
import numpy as np
import faiss

def search_content(query: str, pdf_to_index: pd.DataFrame, k: int = 3):
    # 1. Encodage de la requête textuelle en vecteur numpy
    # On force la sortie en float32 (requis par FAISS) et on ajoute une dimension pour simuler un batch de 1 ligne
    query_vector = model.encode([query], convert_to_numpy=True).astype('float32')

    # 2. Normalisation L2 pour permettre une recherche par similarité cosinus
    faiss.normalize_L2(query_vector)

    # 3. Recherche des k voisins les plus proches dans l'index FAISS
    # sims: distances/similarités (float), ids: indices des lignes correspondantes (int)
    sims, ids = index_content.search(query_vector, k)

    # 4. Filtrage du DataFrame d'origine pour récupérer les lignes trouvées
    results = pdf_to_index[pdf_to_index['id'].isin(ids[0])].copy()

    # 5. Ajout des scores de similarité associés
    results['similarities'] = sims[0]

    return results

# Exécution de la recherche sémantique avec le mot-clé 'animal'
display(search_content('animal', pdf_subset, k=5))

## 🌟 Exercise 4 · ChromaDB collection and querying

In [ ]:
import json
import chromadb
from chromadb.config import Settings

# Initialisation du client
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
collection_name = 'my_news'

# Nettoyage si la collection existe déjà
if any(c.name == collection_name for c in chroma_client.list_collections()):
    chroma_client.delete_collection(name=collection_name)

# 1. TODO: Création de la collection
# Par défaut, ChromaDB utilise le modèle 'all-MiniLM-L6-v2' pour générer les embeddings automatiquement
collection = chroma_client.create_collection(name=collection_name)

# 2. TODO: Ajout des documents dans la collection
# Extraction des titres et des identifiants depuis votre sous-ensemble existant (pdf_subset)
collection.add(
    documents=pdf_subset['title'].tolist(),
    ids=[str(idx) for idx in pdf_subset['id'].tolist()],
    metadatas=[{"source": "newscatcher"} for _ in range(len(pdf_subset))] # Optionnel : métadonnées utiles pour le filtrage
)

# 3. TODO: Exécution de la requête sémantique
# ChromaDB va vectoriser la query à la volée et chercher les 'n_results' voisins les plus proches
results = collection.query(
    query_texts=["animal"],
    n_results=5
)

# Affichage des résultats au format JSON propre
print(json.dumps(results, indent=2))

## 🌟 Exercise 5 · Question answering with a Hugging Face model

In [ ]:
from transformers import pipeline

model_id = 'google/flan-t5-small'  # lightweight, better than tiny GPT-2 for QA

# 1. TODO: Création du pipeline de génération de texte à texte
pipe = pipeline(
    task="text2text-generation",
    model=model_id,
    device=-1 # Utilisez 0 si vous êtes sur Colab avec un GPU (T4) activé pour accélérer l'exécution
)

question = "What's the latest news on space development?"
context_docs = results['documents'][0][:3]
context = ' '.join(context_docs)

# Construction du prompt d'instructions (In-Context Learning)
prompt = f"Answer the question using only the context.\nContext: {context}\nQuestion: {question}\nAnswer:\n"

# 2. Exécution du pipeline sur le prompt
response = pipe(prompt)[0]['generated_text']
print("Réponse générée :")
print(response)